# Calibrating NN Models for Communication Reliability </br>
Date: 13/03/2024

## Imports

In [1]:
import tensorflow as tf
from keras.models import Model
from keras.layers import Dense, Input
from keras import activations
from keras import backend as K
import pandas as pd # for data manipulation 
from scipy.optimize import minimize, differential_evolution
import numpy as np
import glob, math, os
from scipy import special
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

def euclidean_dist(row):
    # Function to calc euclidean distance on every df row 
    euc_dist = math.sqrt(row["U2G_Distance"]**2 - row["Height"]**2)
    return euc_dist

def q_func(x):
    q = 0.5 - 0.5*special.erf(x / np.sqrt(2))
    return q

def friis_calc(P,freq,dist,ple):
    '''
    Friis path loss equation
    P = Tx transmit power
    freq = Signal frequency
    dist = Transmission distance
    ple = Path loss exponent
    '''
    propagation_speed = 299792458
    l = propagation_speed / freq
    h_pl = P * l**2 / (16*math.pi**2)
    P_Rx = h_pl * dist**(-ple)
    return P_Rx

def plos_calc(h_dist, height_tx, height_rx, env='suburban'):
    '''
    % This function implements the LoS probability model from the paper
    % "Blockage Modeling for Inter-layer UAVs Communications in Urban
    % Environments" 
    % param h_dist    : horizontal distance between Tx and Rx (m)
    % param height_tx : height of Tx
    % param height_rx : height of Rx
    '''
    if env == 'suburban':
        a1 = 0.1
        a2 = 7.5e-4
        a3 = 8
    elif env == 'urban':
        a1 = 0.3
        a2 = 5e-4
        a3 = 15
    
    delta_h = height_tx - height_rx
    # pow_factor = 2 * h_dist * math.sqrt(a1*a2/math.pi) + a1 # NOTE: Use this pow_factor if assuming PPP building dist.
    pow_factor = h_dist * math.sqrt(a1*a2) # NOTE: Use this pow_factor if assuming ITU-R assumptions.
    if delta_h == 0:
        p = (1 - math.exp((-(height_tx)**2) / (2*a3**2))) ** pow_factor
    else:
        if delta_h < 0:
            h1 = height_rx
            h2 = height_tx
        else:
            h1 = height_tx
            h2 = height_rx
        delta_h = abs(delta_h)
        p = (1 - (math.sqrt(2*math.pi)*a3 / delta_h) * abs(q_func(h1/a3) - q_func(h2/a3))) ** pow_factor
    return p

def sinr_lognormal_approx(h_dist, height, env='suburban'):
    '''
    To approximate the SNR from signal considering multipath fading and shadowing
    Assuming no interference due to CSMA, and fixed noise
    Inputs:
    h_dist = Horizontal Distance between Tx and Rx
    height = Height difference between Tx and Rx
    env = The operating environment (currently only suburban supported)
    '''
    # Signal properties
    P_Tx_dBm = 20 # Transmit power of 
    P_Tx = 10**(P_Tx_dBm/10) / 1000
    freq = 2.4e9 # Channel frequency (Hz)
    noise_dBm = -86
    noise = 10**(noise_dBm/10) / 1000
    if env == "suburban":
        # ENV Parameters Constants ----------------------------------
        # n_min = 2
        # n_max = 2.75
        # K_dB_min = 7.8
        # K_dB_max = 17.5
        # K_min = 10**(K_dB_min/10)
        # K_max = 10**(K_dB_max/10)
        # alpha = 11.25 # Env parameters for logarithm std dev of shadowing 
        # beta = 0.06 # Env parameters for logarithm std dev of shadowing 
        n_min = 2
        n_max = 2.75
        K_dB_min = 1.4922
        K_dB_max = 12.2272
        K_min = 10**(K_dB_min/10)
        K_max = 10**(K_dB_max/10)
        alpha = 11.1852 # Env parameters for logarithm std dev of shadowing 
        beta = 0.06 # Env parameters for logarithm std dev of shadowing 
        # -----------------------------------------------------------
    elif env == "urban":
        n_min = 1.9
        n_max = 2.7
        K_dB_min = -5
        K_dB_max = 15
        K_min = 10**(K_dB_min/10)
        K_max = 10**(K_dB_max/10)
        alpha = 10.42 # Env parameters for logarithm std dev of shadowing 
        beta = 0.05 # Env parameters for logarithm std dev of shadowing 
    # Calculate fading parameters
    PLoS = plos_calc(h_dist, 0, height, env=env)
    theta_Rx = math.atan2(height, h_dist) * 180 / math.pi # Elevation angle in degrees
    ple = (n_min - n_max) * PLoS + n_max # Path loss exponent
    sigma_phi_dB = alpha*math.exp(-beta*theta_Rx)
    sigma_phi = 10**(sigma_phi_dB/10) # Logarithmic std dev of shadowing
    K = K_min * math.exp(math.log(K_max/K_min) * PLoS**2)
    omega = 1 # Omega of NCS (Rician)
    dist = math.sqrt(h_dist**2 + height**2)
    P_Rx = friis_calc(P_Tx, freq, dist, ple)
    # Approximate L-NCS RV (which is the SNR) as lognormal
    eta = math.log(10) / 10
    mu_phi = 10*math.log10(P_Rx)
    E_phi = math.exp(eta*mu_phi + eta**2*sigma_phi**2/2) # Mean of shadowing RV
    var_phi = math.exp(2*eta*mu_phi+eta**2*sigma_phi**2)*(math.exp(eta**2*sigma_phi**2)-1) # Variance of shadowing RV
    E_chi = (special.gamma(1+1)/(1+K))*special.hyp1f1(-1,1,-K)*omega
    var_chi = (special.gamma(1+2)/(1+K)**2)*special.hyp1f1(-2,1,-K)*omega**2 - E_chi**2
    E_SNR = E_phi * E_chi / noise # Theoretical mean of SINR
    var_SNR = ((var_phi+E_phi**2)*(var_chi+E_chi**2) - E_phi**2 * E_chi**2) / noise**2
    std_dev_SNR = math.sqrt(var_SNR)
    # sigma_ln = math.sqrt(math.log(var_SNR/E_SNR**2 + 1))
    # mu_ln = math.log(E_SNR) - sigma_ln**2/2
    return E_SNR, std_dev_SNR

def normalize_data(df, columns=[], save_details_path=None):
    '''
    columns: The pandas data columns to normalize, given as a list of column names
    '''
    # Define the ranges of parametrers
    max_mean_sinr = 10*math.log10(1123) # The max mean SINR calculated at (0,60) is 1122.743643457063 (linear)
    max_std_dev_sinr = 10*math.log10(466) # The max std dev SINR calculated at (0,60) is 465.2159856885714 (linear)
    min_mean_sinr = 10*math.log10(0.2) # The min mean SINR calculated at (1200,60) is 0.2251212887895188 (linear)
    min_std_dev_sinr = 10*math.log10(0.7) # The min std dev SINR calculated at (1200,300) is 0.7160093126585219 (linear)
    max_height = 300
    min_height = 60
    max_h_dist = 1200
    min_h_dist = 0
    max_mcs = 7
    min_mcs = 0

    # Normalize data (Min Max Normalization between [-1,1])
    if "Height" in columns:
        df["Height"] = df["Height"].apply(lambda x: 2*(x-min_height)/(max_height-min_height) - 1)
    if "U2G_H_Dist" in columns:
        df["U2G_H_Dist"] = df["U2G_H_Dist"].apply(lambda x: 2*(x-min_h_dist)/(max_h_dist-min_h_dist) - 1)
    if "Mean_SINR" in columns:
        df["Mean_SINR"] = df["Mean_SINR"].apply(lambda x: 2*(10*math.log10(x)-min_mean_sinr)/(max_mean_sinr-min_mean_sinr) - 1) # Convert to dB space
    if "Std_Dev_SINR" in columns:
        df["Std_Dev_SINR"] = df["Std_Dev_SINR"].apply(lambda x: 2*(10*math.log10(x)-min_std_dev_sinr)/(max_std_dev_sinr-min_std_dev_sinr) - 1) # Convert to dB space
    if "UAV_Sending_Interval" in columns:
        df["UAV_Sending_Interval"] = df["UAV_Sending_Interval"].replace({10:-1, 20:-0.5, 40:0, 66.7: 0.5, 100:1, 1000:2})
    if "Packet_State" in columns:
        df['Packet_State'] = df['Packet_State'].replace({"Reliable":0, "QUEUE_OVERFLOW":1, "RETRY_LIMIT_REACHED":2, "Delay_Exceeded":3})
    if "Modulation" in columns:
        df['Modulation'] = df['Modulation'].replace({"BPSK":1, "QPSK":0.3333, 16:-0.3333, "QAM-16":-0.3333, "QAM16":-0.3333, 64:-1, "QAM-64":-1, "QAM64":-1})
    if "MCS" in columns:
        df["MCS"] = df["MCS"].apply(lambda x: 2*(x-min_mcs)/(max_mcs-min_mcs) - 1)

    # Record details of inputs and output for model
    if save_details_path is not None:
        f = open(os.path.join(save_details_path,"model_details.txt"), "w")
        f.write("Max Height (m): {}\n".format(max_height))
        f.write("Min Height (m): {}\n".format(min_height))
        f.write("Max H_Dist (m): {}\n".format(max_h_dist))
        f.write("Min H_Dist (m): {}\n".format(min_h_dist))
        f.write("Max Mean SINR (dB): {}\n".format(max_mean_sinr))
        f.write("Min Mean SINR (dB): {}\n".format(min_mean_sinr))
        f.write("Max Std Dev SINR (dB): {}\n".format(max_std_dev_sinr))
        f.write("Min Std Dev SINR (dB): {}\n".format(min_std_dev_sinr))
        f.write("[BPSK: 1, QPSK: 0.3333, QAM16: -0.3333, QAM64: -1]\n")
        f.write("UAV Sending Interval: [10:-1, 20:-0.5, 40:0, 100:0.5, 1000:1]\n")
        f.write("Output: ['Reliable':0, 'QUEUE_OVERFLOW':1, 'RETRY_LIMIT_REACHED':2, 'Delay_Exceeded':3]\n")
        f.close()

    return df

def get_mcs_index(df_in):
    '''
    Gets the MCS index based on modulation and bitrate column of the df_in
    '''
    df = df_in.copy()
    df["MCS"] = ''
    df.loc[(df["Modulation"] == "BPSK") & (df["Bitrate"] == 6.5), "MCS"] = 0 # MCS Index 0
    df.loc[(df["Modulation"] == "QPSK") & (df["Bitrate"] == 13), "MCS"] = 1 # MCS Index 0
    df.loc[(df["Modulation"] == "QPSK") & (df["Bitrate"] == 19.5), "MCS"] = 2 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM16") & (df["Bitrate"] == 26), "MCS"] = 3 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM16") & (df["Bitrate"] == 39), "MCS"] = 4 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 52), "MCS"] = 5 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 58.5), "MCS"] = 6 # MCS Index 0
    df.loc[(df["Modulation"] == "QAM64") & (df["Bitrate"] == 65), "MCS"] = 7 # MCS Index 0

    return df

def get_output_layer(model, layer_name):
    # From https://github.com/jacobgil/keras-cam/blob/master/model.py#L79
    # get the symbolic outputs of each "key" layer (we gave them unique names).
    layer_dict = dict([(layer.name, layer) for layer in model.layers])
    layer = layer_dict[layer_name]
    return layer

2024-04-24 11:43:28.097707: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-24 11:43:28.265196: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-04-24 11:43:28.270636: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-04-24 11:43:28.270652: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if yo

## Get Weights of Trained Model

In [2]:
model = tf.keras.models.load_model("/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/nn_checkpoints/djispark_nnv4_wobn_dl_05042024/model.020-0.2034.h5", compile=False)
model.compile(optimizer='adam', 
              loss={'packet_state': 'categorical_crossentropy'},
              metrics={'packet_state': 'accuracy'})

2024-04-24 11:43:53.318399: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2024-04-24 11:43:53.318451: W tensorflow/stream_executor/cuda/cuda_driver.cc:263] failed call to cuInit: UNKNOWN ERROR (303)
2024-04-24 11:43:53.318478: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (MMEN4-0049-RE): /proc/driver/nvidia/version does not exist
2024-04-24 11:43:53.318777: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Rebuild Model with No SoftMax Activation

In [3]:
def build_nn_model_v4_wobatchnorm_noactivation():
    # For multiple output model
    # Version 4: Having only a single output layer for packet state
    inputs = Input(shape=(4,))
    base = Dense(100, activation='relu')(inputs)
    base = Dense(50, activation='relu')(base)
    base = Dense(25, activation='relu')(base)
    base = Dense(10, activation='relu')(base)
    packet_state_out = Dense(4, activation=None, name='packet_state_no_activation')(base)
    model = Model(inputs=inputs, outputs = packet_state_out)
    return model

model_no_act = build_nn_model_v4_wobatchnorm_noactivation()
model_no_act.set_weights(model.get_weights())

## Get logits from train dataset

In [4]:
DATASET_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/data_processed/DJI_Spark_Downlink_Reliability.csv"
# DATASET_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJIMavicAir/Test_Dataset_1_NP10000_DJIMavicAir_Uplink_Reliability.csv"
SAVE_PATH = "/home/research-student/omnet-fanet/data-processing-scripts/"
X_TRAIN_FILENAME = "NN_Temp_Scaling_X_train.npy"
Y_TRAIN_FILENAME = "NN_Temp_Scaling_y_train.npy"
# X_TRAIN_FILENAME = "NN_Temp_Scaling_X_test.npy"
# Y_TRAIN_FILENAME = "NN_Temp_Scaling_y_test.npy"
RELIABILITY_TH = 0.99
MIN_FAILURE_PROB = 0.01

# Load train dataset
df_dtypes = {"Horizontal_Distance": np.float64, "Height": np.int16,	"U2G_Distance": np.int32, "UAV_Sending_Interval": np.float64, "Mean_SINR": np.float64, "Std_Dev_SINR": np.float64,
                 "Num_Sent": np.int32, "Num_Reliable": np.int32, "Num_Delay_Excd": np.int32, "Num_Incr_Rcvd": np.int32, "Num_Q_Overflow": np.int32, "Modulation": str, "Bitrate": np.float64}
dataset_details_df = pd.read_csv(DATASET_PATH, 
                                usecols = ["Mean_SINR", "Std_Dev_SINR", "UAV_Sending_Interval", "Modulation", "Bitrate", "Num_Sent", "Num_Reliable", "Num_Delay_Excd",
                                            "Num_Incr_Rcvd", "Num_Q_Overflow"],
                                dtype=df_dtypes)
dataset_details_df = get_mcs_index(dataset_details_df)
dataset_details_df = normalize_data(dataset_details_df, columns=["Mean_SINR", "Std_Dev_SINR", "UAV_Sending_Interval", "MCS"], save_details_path=None) 
dataset_details_df["Reliability"] = (dataset_details_df["Num_Reliable"] / dataset_details_df["Num_Sent"]).values
dataset_details_df["Delay_Excd_Prob"] = (dataset_details_df["Num_Delay_Excd"] / dataset_details_df["Num_Sent"]).values
dataset_details_df["Queue_Overflow_Prob"] = (dataset_details_df["Num_Q_Overflow"] / dataset_details_df["Num_Sent"]).values
dataset_details_df["Incr_Rcvd_Prob"] = (dataset_details_df["Num_Incr_Rcvd"] / dataset_details_df["Num_Sent"]).values
dataset_details_df["Failure_Mode"] = dataset_details_df[["Queue_Overflow_Prob", "Incr_Rcvd_Prob", "Delay_Excd_Prob"]].idxmax(axis=1)
dataset_details_df.loc[(dataset_details_df["Queue_Overflow_Prob"] < MIN_FAILURE_PROB) & (dataset_details_df["Incr_Rcvd_Prob"] < MIN_FAILURE_PROB) 
                    & (dataset_details_df["Delay_Excd_Prob"] < MIN_FAILURE_PROB),["Failure_Mode"]] = "None"

# Get logits from model, output of reliable state (reliability>=RELIABILITY_TH)
X_train = model_no_act.predict(dataset_details_df[["Mean_SINR", "Std_Dev_SINR", "UAV_Sending_Interval", "MCS"]].values)
# y_train = dataset_details_df["Reliability"] >= RELIABILITY_TH
# y_train = dataset_details_df["Reliability"].to_numpy()
y_train = dataset_details_df["Failure_Mode"].replace({"Queue_Overflow_Prob":1, "Incr_Rcvd_Prob":2, "Delay_Excd_Prob":3, "None":4})
# Save inputs and output
np.save(os.path.join(SAVE_PATH, X_TRAIN_FILENAME), X_train)
np.save(os.path.join(SAVE_PATH, Y_TRAIN_FILENAME), y_train.to_numpy())


1089/1089 [==============================] - 1s 986us/step


## Calibrate NN using Temperature Scaling

In [23]:
SAVE_PATH = "/home/research-student/omnet-fanet/data-processing-scripts/"
X_TRAIN_FILENAME = "NN_Temp_Scaling_X_train.npy"
Y_TRAIN_FILENAME = "NN_Temp_Scaling_y_train.npy"
RELIABILITY_TH = 0.99
SEED = 100
MIN_FAILURE_PROB = 0.01

X_train = np.load(os.path.join(SAVE_PATH, X_TRAIN_FILENAME))
y_train = np.load(os.path.join(SAVE_PATH, Y_TRAIN_FILENAME))

def objective_accuracy(T, X_train, y_train, reliability_th):
    """
    Objective function for temperature scaling optimization
    T: temperature to scale the logits
    X_train: Logits from NN model evaluated on dataset (numpy arr)
    y_train: Boolean array of True/False for reliability state (reliability>=threshold) (numpy arr)
    reliability_th: Threshold value for reliability
    """
    print(T)
    calibrated_reliability_prediction = np.array([activations.softmax(K.constant([logits/T]), axis=-1)[0][0].numpy() for logits in X_train])
    calibrated_reliability_state = calibrated_reliability_prediction >= reliability_th
    accuracy = accuracy_score(calibrated_reliability_state, y_train)
    # Return the complement of accuracy since we are using a minimizing optimization
    return 1 - accuracy

def objective_failure_mode_classification_accuracy(T, X_train, y_train, min_failure_prob):
    """
    Objective function for temperature scaling optimization for classification accuracy
    T: temperature to scale the logits
    X_train: Logits from NN model evaluated on dataset (numpy arr)
    y_train: pandas series on failure modes (from pd.replace())
    min_failure_prob: Minimum probability of failure, else failure mode is None
    """
    print(T)
    calib_nn_prediction = [activations.softmax(K.constant([logits/T]), axis=-1)[0].numpy() for logits in X_train]
    temp_df  =pd.DataFrame()
    temp_df['CalibNN_Predicted_Reliability'] = [prob[0] for prob in calib_nn_prediction]
    temp_df['CalibNN_Predicted_Queue_Overflow_Prob'] = [prob[1] for prob in calib_nn_prediction]
    temp_df['CalibNN_Predicted_Incr_Rcvd_Prob'] = [prob[2] for prob in calib_nn_prediction]
    temp_df['CalibNN_Predicted_Delay_Excd_Prob'] = [prob[3] for prob in calib_nn_prediction]
    temp_df["CalibNN_Predicted_Failure_Mode"] = temp_df[["CalibNN_Predicted_Queue_Overflow_Prob", "CalibNN_Predicted_Incr_Rcvd_Prob", "CalibNN_Predicted_Delay_Excd_Prob"]].idxmax(axis=1)
    temp_df.loc[(temp_df["CalibNN_Predicted_Queue_Overflow_Prob"] < min_failure_prob) & (temp_df["CalibNN_Predicted_Incr_Rcvd_Prob"] < min_failure_prob) 
                            & (temp_df["CalibNN_Predicted_Delay_Excd_Prob"] < min_failure_prob),["CalibNN_Predicted_Failure_Mode"]] = "None"
    calib_nn_failure_mode_predicted = temp_df["CalibNN_Predicted_Failure_Mode"].replace({"CalibNN_Predicted_Queue_Overflow_Prob":1, "CalibNN_Predicted_Incr_Rcvd_Prob":2, "CalibNN_Predicted_Delay_Excd_Prob":3, "None":4})
    # print(temp_df.head())
    # print(calib_nn_failure_mode_predicted)
    accuracy = accuracy_score(y_train, calib_nn_failure_mode_predicted)
    print(accuracy)
    # Return the complement of classification accuracy since we are using a minimizing optimization
    return 1 - accuracy

init_T = 1
bound_T = (0.3, 1.7)
# calibrated_T = minimize(objective, init_T, args=(X_train, y_train, RELIABILITY_TH), method='BFGS', options={'gtol': 1e-3, 'eps': 0.1, 'maxiter': 100, 'disp': True})
# calibrated_T = differential_evolution(objective_accuracy, bounds=[bound_T], args=(X_train, y_train, RELIABILITY_TH), workers=1, seed=SEED, disp=True)
calibrated_T = differential_evolution(objective_failure_mode_classification_accuracy, bounds=[bound_T], args=(X_train, y_train, MIN_FAILURE_PROB), workers=16, seed=SEED, disp=True)

/home/research-student/tf_venv/lib/python3.8/site-packages/scipy/optimize/_differentialevolution.py:382: UserWarning: differential_evolution: the 'workers' keyword has overridden updating='immediate' to updating='deferred'
  with DifferentialEvolutionSolver(func, bounds, args=args,


[0.35071779][0.52628831]

[1.62717177][1.03041292]

[1.31652338][0.92260325][1.19367538]


[1.05942595][1.52344851][0.65884577]

[0.77801312]
[0.41931448]
[0.67377376]
[1.4372973]
[1.3461922]



0.905274334251607
0.9206554178145088
0.9159205693296603
0.9393939393939394
0.9149735996326905
0.9356921487603306
0.9122474747474747
0.9319329660238751
0.9030073461891643
0.9037534435261708
0.9263085399449036
0.9275424701561065
0.9073691460055097
0.9092917814508723
0.9076561065197429
[1.43371944][1.36953452][0.92029694]

[1.44608041]
[1.56630846]
[1.24824445][1.45717872][1.05954147]

[1.52864323]

[0.45293744]
[1.6371445]
[1.37473255]

[0.84387177][1.65854938][1.24788578]


0.9402548209366391
0.9106691919191919
0.9319042699724518
0.9287477043158862
0.9396522038567493
0.923151974288338
0.9289485766758494
0.923151974288338
0.9358643250688705
0.9122474747474747
0.932363406795225
0.9375573921028466
0.9043273645546372
0.9328512396694215
0.9159205693296603
[1.4095135]
[1.52838562]
[1.641582][1.26153497]

[1.31753527][1.59248166]
[1.61329821]
[0.97157127]

[1.514006]
[0.81471941]
[1.47595814]
[0.93582897]
[0.74991217]
[0.42980781]
[0.3846131]
differential_evolution step 1: f(x)= 0.0597452
0.92

In [28]:
objective_failure_mode_classification_accuracy(2, X_train, y_train, MIN_FAILURE_PROB)

2


0.9462522956841138


0.05374770431588616

In [24]:
calibrated_T.x

array([1.6988107])

In [6]:
calibrated_reliability_prediction = np.array([activations.softmax(K.constant([logits/1.10233671]), axis=-1)[0][0].numpy() for logits in X_train])
calibrated_reliability_state = calibrated_reliability_prediction >= RELIABILITY_TH
accuracy_score(calibrated_reliability_state, y_train)

0.9953225436179982

In [130]:
calibrated_reliability_prediction = np.array([activations.softmax(K.constant([logits/1.06709991]), axis=-1)[0][0].numpy() for logits in X_train])
calibrated_reliability_state = calibrated_reliability_prediction >= RELIABILITY_TH
accuracy_score(calibrated_reliability_state, y_train)

0.9944903581267218

## Validate that model_no_act produces same results before calibration

In [29]:
# Set minimum probability for failure mode to be considered
MIN_FAILURE_PROB = 0.01

RELIABILITY_TH = [0.95, 0.99, 0.999] # For calculation max AE in reliable region

DATASET_PATH = "/media/research-student/One Touch/FANET_Dataset/Dataset_NP10000_DJISpark/test_dataset_2_processed/Downlink_Reliability.csv"
test_data_df = pd.read_csv(DATASET_PATH)
test_data_df = get_mcs_index(test_data_df)
test_data_df = normalize_data(test_data_df, columns=["Mean_SINR", "Std_Dev_SINR", "UAV_Sending_Interval", "MCS"], save_details_path=None)

test_data_df["Reliability"] = (test_data_df["Num_Reliable"] / test_data_df["Num_Sent"]).values
test_data_df["Delay_Excd_Prob"] = (test_data_df["Num_Delay_Excd"] / test_data_df["Num_Sent"]).values
test_data_df["Queue_Overflow_Prob"] = (test_data_df["Num_Q_Overflow"] / test_data_df["Num_Sent"]).values
test_data_df["Incr_Rcvd_Prob"] = (test_data_df["Num_Incr_Rcvd"] / test_data_df["Num_Sent"]).values

prediction = model_no_act.predict(test_data_df[["Mean_SINR", "Std_Dev_SINR", "UAV_Sending_Interval", "MCS"]].values)

# Save the results to CSV
T = 2 # Temperature of calibration
# T = 1 # Temperature of calibration
calib_nn_prediction = [activations.softmax(K.constant([logits/T]), axis=-1)[0].numpy() for logits in prediction]
test_data_df['Predicted_Reliability'] = [prob[0] for prob in calib_nn_prediction]
test_data_df['Predicted_Queue_Overflow_Prob'] = [prob[1] for prob in calib_nn_prediction]
test_data_df['Predicted_Incr_Rcvd_Prob'] = [prob[2] for prob in calib_nn_prediction]
test_data_df['Predicted_Delay_Excd_Prob'] = [prob[3] for prob in calib_nn_prediction]

test_data_df["Reliability_Class"] = pd.cut(test_data_df["Reliability"], bins=[-0.1,0.1,0.5,0.9,1], labels=["Low", "ModeratelyLow", "ModeratelyHigh", "High"])
test_data_df["Predicted_Reliability_Class"] = pd.cut(test_data_df["Predicted_Reliability"], bins=[-0.1,0.1,0.5,0.9,1], labels=["Low", "ModeratelyLow", "ModeratelyHigh", "High"])
test_data_df["Failure_Mode"] = test_data_df[["Queue_Overflow_Prob", "Incr_Rcvd_Prob", "Delay_Excd_Prob"]].idxmax(axis=1)
test_data_df["Predicted_Failure_Mode"] = test_data_df[["Predicted_Queue_Overflow_Prob", "Predicted_Incr_Rcvd_Prob", "Predicted_Delay_Excd_Prob"]].idxmax(axis=1)
# Replace label for Failure Mode with "None" if none of the failure modes have a probability > 5%
test_data_df.loc[(test_data_df["Queue_Overflow_Prob"] < MIN_FAILURE_PROB) & (test_data_df["Incr_Rcvd_Prob"] < MIN_FAILURE_PROB) & (test_data_df["Delay_Excd_Prob"] < MIN_FAILURE_PROB),["Failure_Mode"]] = "None"
test_data_df.loc[(test_data_df["Predicted_Queue_Overflow_Prob"] < MIN_FAILURE_PROB) & (test_data_df["Predicted_Incr_Rcvd_Prob"] < MIN_FAILURE_PROB) & (test_data_df["Predicted_Delay_Excd_Prob"] < MIN_FAILURE_PROB),["Predicted_Failure_Mode"]] = "None"

# Compute the model accuracy and mean abs err
failure_mode = test_data_df["Failure_Mode"].replace({"Queue_Overflow_Prob":1, "Incr_Rcvd_Prob":2, "Delay_Excd_Prob":3, "None":4})
failure_mode_predicted = test_data_df["Predicted_Failure_Mode"].replace({"Predicted_Queue_Overflow_Prob":1, "Predicted_Incr_Rcvd_Prob":2, "Predicted_Delay_Excd_Prob":3, "None":4})
reliability_accuracy = accuracy_score(test_data_df["Reliability_Class"], test_data_df["Predicted_Reliability_Class"])
failure_mode_accuracy = accuracy_score(failure_mode, failure_mode_predicted)
reliability_mae = np.mean(abs(test_data_df['Reliability'].values - test_data_df['Predicted_Reliability'].values))
queue_overflow_mae = np.mean(abs(test_data_df['Queue_Overflow_Prob'].values - test_data_df['Predicted_Queue_Overflow_Prob'].values))
incr_rcvd_mae = np.mean(abs(test_data_df['Incr_Rcvd_Prob'].values - test_data_df['Predicted_Incr_Rcvd_Prob'].values))
delay_excd_mae = np.mean(abs(test_data_df['Delay_Excd_Prob'].values - test_data_df['Predicted_Delay_Excd_Prob'].values))
reliability_maxae = np.max(abs(test_data_df['Reliability'].values - test_data_df['Predicted_Reliability'].values))
queue_overflow_maxae = np.max(abs(test_data_df['Queue_Overflow_Prob'].values - test_data_df['Predicted_Queue_Overflow_Prob'].values))
incr_rcvd_maxae = np.max(abs(test_data_df['Incr_Rcvd_Prob'].values - test_data_df['Predicted_Incr_Rcvd_Prob'].values))
delay_excd_maxae = np.max(abs(test_data_df['Delay_Excd_Prob'].values - test_data_df['Predicted_Delay_Excd_Prob'].values))

# Print results
print("Reliability - Accuracy: {}, MAE: {}, MaxAE: {}".format(reliability_accuracy, reliability_mae, reliability_maxae))
print("Failure Mode - Accuracy: {}".format(failure_mode_accuracy))
print("Queue Overflow - MeanAE: {}, MaxAE: {}".format(queue_overflow_mae, queue_overflow_maxae))
print("Incorrectly Received - MeanAE: {}, MaxAE: {}".format(incr_rcvd_mae, incr_rcvd_maxae))
print("Delay Exceeded - MeanAE: {}, MaxAE: {}".format(delay_excd_mae, delay_excd_maxae))
print("Average Failure Mode Mean AE: {}".format(np.mean([queue_overflow_mae, incr_rcvd_mae, delay_excd_mae])))

for reliability_th in RELIABILITY_TH:
    # Get the Max Abs Err of reliability, but only when either the simulated/predicted reliability is above the threshold
    # test_data_reliable_df = test_data_df.loc[(test_data_df["Reliability"]>=reliability_th) | (test_data_df["Predicted_Reliability"]>=reliability_th)]
    # rel_err = test_data_reliable_df['Reliability'].values - test_data_reliable_df['Predicted_Reliability'].values
    # reliability_maxae_reliable = np.max(abs(rel_err))
    # reliability_mean_reliable = np.mean(abs(rel_err))
    # print("Reliability - MeanAE_Reliable_Region: {}, MaxAE_Reliable_Region: {}".format(reliability_mean_reliable, reliability_maxae_reliable))

    # UNCOMMENT TO EVALUATE reliability_state_accuracy over reliable/predicted_reliable region only
    # Get the accuracy of predicting reliability above the threshold
    # test_data_df["Reliable_State"] = test_data_df["Reliability"] >= reliability_th
    # test_data_df["Predicted_Reliable_State"] = test_data_df["Predicted_Reliability"] >= reliability_th
    # reliability_state_accuracy = accuracy_score(test_data_df["Reliable_State"], test_data_df["Predicted_Reliable_State"])

    # # UNCOMMENT TO EVALUATE reliability_state_accuracy over entire region
    # # Get the accuracy of predicting reliability above the threshold
    test_data_df["Reliable_State"] = test_data_df["Reliability"] >= reliability_th
    test_data_df["Predicted_Reliable_State"] = test_data_df["Predicted_Reliability"] >= reliability_th
    reliability_state_accuracy = accuracy_score(test_data_df["Reliable_State"], test_data_df["Predicted_Reliable_State"])
    print("Reliability - Accuracy_Reliable_Region >= {}: {}".format(reliability_th, reliability_state_accuracy))


# test_data_df.to_csv("Test_Model_No_Act.csv")
# test_data_df.to_csv("Test_Dataset_2_NP10000_DJIMavicAir_Video_Reliability_Calibrated_Results.csv")

 78/120 [==================>...........] - ETA: 0s

120/120 [==============================] - 0s 1ms/step
Reliability - Accuracy: 0.9846354166666667, MAE: 0.008944325650151509, MaxAE: 0.39395209159851075
Failure Mode - Accuracy: 0.9486979166666667
Queue Overflow - MeanAE: 0.12060183239407407, MaxAE: 0.21971132793426518
Incorrectly Received - MeanAE: 0.0898678777731597, MaxAE: 0.14806588785648345
Delay Exceeded - MeanAE: 0.036402239348036956, MaxAE: 0.4168200266361236
Average Failure Mode Mean AE: 0.08229064983842357
Reliability - Accuracy_Reliable_Region >= 0.95: 0.9815104166666667
Reliability - Accuracy_Reliable_Region >= 0.99: 0.95078125
Reliability - Accuracy_Reliable_Region >= 0.999: 0.9651041666666667


In [32]:
t1 = test_data_reliable_df["Reliability"].to_numpy() >= RELIABILITY_TH
t2 = test_data_reliable_df["Predicted_Reliability"].to_numpy() >= RELIABILITY_TH
accuracy_score(t1,t2)

0.9674267100977199